# Proyecto de Data Mining — Hito 1
## Perfiles de riesgo nutricional y de anemia en niños menores de 5 años (ENDES 2025)

**Curso:** Data Mining — Universidad del Pacífico
**Docente:** Soledad Espezúa (s.espezual@up.edu.pe)
**Autor:** Mateo

---

### Contexto y problema

La anemia infantil es uno de los problemas de salud pública más persistentes en el Perú, con brechas importantes entre regiones y entre zonas urbanas y rurales. Este proyecto busca caracterizar el estado nutricional y de anemia de niños menores de 5 años a partir de la Encuesta Demográfica y de Salud Familiar (ENDES) 2025, incorporando además el contexto de sus madres (edad, hemoglobina, nivel educativo), para identificar perfiles de riesgo que puedan orientar la priorización de intervenciones de salud (suplementación de hierro, seguimiento nutricional, programas alimentarios).

**A quién beneficiaría este análisis:** entidades de salud pública (MINSA, gobiernos regionales de salud) y programas sociales orientados a primera infancia, que necesitan priorizar recursos limitados hacia los grupos de mayor riesgo en lugar de aplicar intervenciones genéricas a toda la población infantil.

**Objetivo de análisis:** este es un problema de **segmentación** — se busca agrupar niños según la combinación de sus indicadores antropométricos y de hemoglobina, para descubrir perfiles de riesgo que no son evidentes mirando cada variable por separado.

**Unidad de análisis:** cada fila de la base final representa **un niño menor de 5 años, vinculado a los datos de su madre** (par madre-hijo).

### Fuentes de datos

| | RECH5 — Mujeres de 12 a 49 años | RECH6 — Niños menores de 5 años |
|---|---|---|
| **Dataset** | Cuestionario del Hogar, módulo de mujeres (RECH5) | Cuestionario del Hogar, módulo de niños (RECH6) |
| **Institución responsable** | Instituto Nacional de Estadística e Informática (INEI) — Encuesta Demográfica y de Salud Familiar (ENDES) 2025 | INEI — ENDES 2025 |
| **Enlace de acceso** | https://proyectos.inei.gob.pe/microdatos/consulta.asp?cmbencuesta=Encuesta+Demogr%E1fica+y+de+Salud+Familiar+-+ENDES&cmbanno=2025&cmbTrimestre=5|
| **Ruta de navegación** | En el link se ubica el módulo correspondiente a mujeres (RECH5) en la lista de módulos disponibles → se descarga en formato CSV. | Misma ruta que RECH5, seleccionando el módulo de niños (RECH6) en vez de mujeres. |
| **Variables principales usadas** | Edad, peso, talla, nivel de hemoglobina, nivel de anemia, nivel educativo | Edad en meses, peso, talla, sexo, nivel de hemoglobina, nivel de anemia |
| **Forma de acceso** | Descarga directa en CSV, de libre acceso, sin registro previo | Descarga directa en CSV, de libre acceso, sin registro previo |

Ambas fuentes comparten el diccionario oficial de variables (`Diccionario_-_RECH5.pdf`, `Diccionario_-_RECH6.pdf`), publicado por el INEI junto con cada módulo, que documenta los códigos de captura y los valores sentinela usados para "no medido".

In [28]:
import pandas as pd
import numpy as np
import plotly.express as px

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 80)

## 1. Carga de datos

Cargamos ambas fuentes y seleccionamos, desde el inicio, el subconjunto de columnas relevante para el problema, evitando arrastrar las columnas originales de cada archivo cuando la mayoría son percentiles/desviaciones redundantes, variables de VIH, consentimientos o metadata de la encuesta que no aportan al objetivo del proyecto.


In [29]:
madres_raw = pd.read_csv("RECH5_2025.csv", low_memory=False)
ninos_raw = pd.read_csv("RECH6_2025.csv", low_memory=False)

cols_madres = ["Año","Identificación Cuestionario Individual","Número de orden en el hogar",
    "Edad de la madre en años","Peso de la madre","Talla de la madre",
    "Resultado de medición de la madre","Nivel de hemoglobina de la madre",
    "Nivel de Anemia NUEVA DIRECTRIZ OMS 2024/RM 251-2024-MINSA (madre)",
    "Nivel educativo más alto de la madre"]

cols_ninos = ["Año","Identificación Cuestionario Individual","Número de orden en el hogar del niño",
    "Edad en meses del niño","Peso en kilogramos del niño","Altura en centímetros del niño",
    "El niño se midió acostado o de pie","Sexo del niño","Nivel de hemoglobina del niño",
    "Nivel de Anemia NUEVA DIRECTRIZ del niño","Número de orden de la madre en el hogar"]

madres = madres_raw[cols_madres].rename(columns={
    "Nivel de Anemia NUEVA DIRECTRIZ OMS 2024/RM 251-2024-MINSA (madre)": "Nivel de anemia de la madre",
    "Nivel educativo más alto de la madre": "Nivel educativo de la madre"
}).copy()

ninos = ninos_raw[cols_ninos].rename(columns={
    "Nivel de Anemia NUEVA DIRECTRIZ del niño": "Nivel de anemia del niño"
}).copy()

print("madres (RECH5):", madres.shape)
print("ninos (RECH6):", ninos.shape)
display(madres.head())
display(ninos.head())

madres (RECH5): (37679, 10)
ninos (RECH6): (19298, 11)


,Año,Identificación Cuestionario Individual,Número de orden en el hogar,Edad de la madre en años,Peso de la madre,Talla de la madre,Resultado de medición de la madre,Nivel de hemoglobina de la madre,Nivel de anemia de la madre,Nivel educativo de la madre
0,2025,434400501,2,26,586,1545,0,151,4,2
1,2025,434401201,1,38,504,1517,0,128,3,3
2,2025,434402601,1,44,571,1487,0,154,4,1
3,2025,434402601,2,22,408,1484,0,118,2,2
4,2025,434402601,3,13,337,1439,0,148,4,1


,Año,Identificación Cuestionario Individual,Número de orden en el hogar del niño,Edad en meses del niño,Peso en kilogramos del niño,Altura en centímetros del niño,El niño se midió acostado o de pie,Sexo del niño,Nivel de hemoglobina del niño,Nivel de anemia del niño,Número de orden de la madre en el hogar
0,2025,434400501,3,29,131,894,2,2,127,4,2
1,2025,434401201,2,13,114,761,1,1,123,4,1
2,2025,434402601,4,45,139,959,2,2,140,4,1
3,2025,434403101,3,37,149,958,2,1,128,4,993
4,2025,434407401,7,13,81,703,1,2,124,4,2


## 2. Inspección inicial

Tamaño, tipos de datos y resumen estadístico de cada fuente, antes de cualquier limpieza.

In [30]:
print("Tipos de datos detectados en madres:")
display(madres.dtypes)

print("Tipos de datos detectados en niños:")
display(ninos.dtypes)

Tipos de datos detectados en madres:


,0
Año,int64
Identificación Cuestionario Individual,int64
Número de orden en el hogar,int64
Edad de la madre en años,int64
Peso de la madre,object
Talla de la madre,object
Resultado de medición de la madre,int64
Nivel de hemoglobina de la madre,int64
Nivel de anemia de la madre,int64
Nivel educativo de la madre,int64


Tipos de datos detectados en niños:


,0
Año,int64
Identificación Cuestionario Individual,int64
Número de orden en el hogar del niño,int64
Edad en meses del niño,int64
Peso en kilogramos del niño,int64
Altura en centímetros del niño,int64
El niño se midió acostado o de pie,object
Sexo del niño,int64
Nivel de hemoglobina del niño,int64
Nivel de anemia del niño,int64


In [31]:
print("Resumen de madres:")
display(madres.describe(include="all").T)

print("Resumen de niños:")
display(ninos.describe(include="all").T)

Resumen de madres:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Año,37679.0,NaN,NaN,NaN,2025.0,0.0,2025.0,2025.0,2025.0,2025.0,2025.0
Identificación Cuestionario Individual,37679.0,NaN,NaN,NaN,599658818.851562,94316195.361607,434400501.0,517802201.0,600206301.0,681510051.0,760118301.0
Número de orden en el hogar,37679.0,NaN,NaN,NaN,2.551952,1.446188,1.0,2.0,2.0,3.0,17.0
Edad de la madre en años,37679.0,NaN,NaN,NaN,29.074259,10.22965,12.0,20.0,29.0,37.0,49.0
Peso de la madre,37679,926,9999,4683,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Talla de la madre,37679,438,9999,4684,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Resultado de medición de la madre,37679.0,NaN,NaN,NaN,0.453781,1.228253,0.0,0.0,0.0,0.0,6.0
Nivel de hemoglobina de la madre,37679.0,NaN,NaN,NaN,244.348921,290.035745,50.0,124.0,135.0,152.0,999.0
Nivel de anemia de la madre,37679.0,NaN,NaN,NaN,4.333581,1.888993,1.0,4.0,4.0,4.0,9.0
Nivel educativo de la madre,37679.0,NaN,NaN,NaN,1.960217,0.619356,0.0,2.0,2.0,2.0,8.0


Resumen de niños:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Año,19298.0,NaN,NaN,NaN,2025.0,0.0,2025.0,2025.0,2025.0,2025.0,2025.0
Identificación Cuestionario Individual,19298.0,NaN,NaN,NaN,601533002.021401,94751977.211817,434400501.0,519809926.0,602754251.0,685009551.0,760118301.0
Número de orden en el hogar del niño,19298.0,NaN,NaN,NaN,4.348015,1.533829,2.0,3.0,4.0,5.0,25.0
Edad en meses del niño,19298.0,NaN,NaN,NaN,30.712354,16.767492,0.0,16.0,31.0,45.0,59.0
Peso en kilogramos del niño,19298.0,NaN,NaN,NaN,252.703182,1094.44287,22.0,102.0,130.0,156.0,9999.0
Altura en centímetros del niño,19298.0,NaN,NaN,NaN,994.087677,1056.937793,367.0,774.0,894.0,983.0,9999.0
El niño se midió acostado o de pie,19298,3,2,11989,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sexo del niño,19298.0,NaN,NaN,NaN,1.484092,0.49976,1.0,1.0,1.0,2.0,2.0
Nivel de hemoglobina del niño,19298.0,NaN,NaN,NaN,196.629858,248.289977,50.0,112.0,121.0,133.0,999.0
Nivel de anemia del niño,19298.0,NaN,NaN,NaN,4.093222,1.626708,1.0,3.0,4.0,4.0,9.0


## 3. Diagnóstico preliminar de calidad

### 3.1 Valores faltantes

En ENDES, los valores faltantes casi nunca aparecen como `NaN` se codifican como valores numéricos sentinela (`9999`, `999`, etc.) según el diccionario oficial. Por eso, un `isna()` directo es engañoso: las fuentes están casi completas cuando en realidad hay una cantidad significativa de "no medidos" escondida dentro de columnas aparentemente numéricas.

In [32]:
print("Valores faltantes (NaN reales) en madres:")
display(madres.isna().sum())

print("Valores faltantes (NaN reales) en niños:")
display(ninos.isna().sum())

# Variables numéricas quedaron como texto -> forzamos su tipo antes de seguir
num_madres = ["Edad de la madre en años","Peso de la madre","Talla de la madre",
              "Nivel de hemoglobina de la madre","Nivel de anemia de la madre","Nivel educativo de la madre"]
num_ninos = ["Edad en meses del niño","Peso en kilogramos del niño","Altura en centímetros del niño",
             "Nivel de hemoglobina del niño","Nivel de anemia del niño"]

for c in num_madres:
    madres[c] = pd.to_numeric(madres[c], errors="coerce")
for c in num_ninos:
    ninos[c] = pd.to_numeric(ninos[c], errors="coerce")

print("\nFaltante oculto tras el código sentinela — ej. Peso de la madre (9999 = no medido):")
print((madres["Peso de la madre"] == 9999).sum(), "casos")
print("\nEj. Nivel de hemoglobina del niño (999 = no medido):")
print((ninos["Nivel de hemoglobina del niño"] == 999).sum(), "casos")

Valores faltantes (NaN reales) en madres:


,0
Año,0
Identificación Cuestionario Individual,0
Número de orden en el hogar,0
Edad de la madre en años,0
Peso de la madre,0
Talla de la madre,0
Resultado de medición de la madre,0
Nivel de hemoglobina de la madre,0
Nivel de anemia de la madre,0
Nivel educativo de la madre,0


Valores faltantes (NaN reales) en niños:


,0
Año,0
Identificación Cuestionario Individual,0
Número de orden en el hogar del niño,0
Edad en meses del niño,0
Peso en kilogramos del niño,0
Altura en centímetros del niño,0
El niño se midió acostado o de pie,0
Sexo del niño,0
Nivel de hemoglobina del niño,0
Nivel de anemia del niño,0



Faltante oculto tras el código sentinela — ej. Peso de la madre (9999 = no medido):
4683 casos

Ej. Nivel de hemoglobina del niño (999 = no medido):
1682 casos


### 3.2 Variables categóricas y errores de registro

In [33]:
print("Valores de Sexo del niño:")
display(ninos["Sexo del niño"].value_counts(dropna=False))

print("Valores de El niño se midió acostado o de pie:")
display(ninos["El niño se midió acostado o de pie"].value_counts(dropna=False))

print("Valores de Resultado de medición de la madre:")
display(madres["Resultado de medición de la madre"].value_counts(dropna=False))

print("Valores de Nivel de anemia del niño:")
display(ninos["Nivel de anemia del niño"].value_counts(dropna=False))

Valores de Sexo del niño:


,count
Sexo del niño,
1,9956
2,9342


Valores de El niño se midió acostado o de pie:


,count
El niño se midió acostado o de pie,
2,11989
1,7061
,248


Valores de Resultado de medición de la madre:


,count
Resultado de medición de la madre,
0,32993
4,2464
3,2029
6,190
5,3


Valores de Nivel de anemia del niño:


,count
Nivel de anemia del niño,
4,12268
3,4107
9,1682
2,1219
1,22


**Lo que se observa:** `Sexo del niño` está limpio (solo dos códigos). `El niño se midió acostado o de pie` trae un texto vacío como si fuera una tercera categoría este es un faltante disfrazado, no una categoría real. `Nivel de anemia del niño` trae un código `9` que no corresponde a ninguna de las 4 categorías documentadas (Grave/Moderada/Leve/Sin anemia), también se trata como faltante.

## 4. Clave de integración y validación de duplicados

No existe un identificador único de persona compartido entre ambas fuentes. Cada persona se ubica combinando:

* `Identificación Cuestionario Individual` identifica el hogar/cuestionario, igual en ambas fuentes.
* La posición de la persona dentro del hogar: `Número de orden en el hogar` (la madre, en RECH5) debe coincidir con `Número de orden de la madre en el hogar` (registrado en la ficha de cada niño, en RECH6).

Por eso la clave de integración es **compuesta**: usar solo el identificador de hogar generaría un cruce cartesiano entre todas las mujeres y todos los niños del mismo hogar, no un emparejamiento madre-hijo real.

Antes de integrar, validamos que la clave **propia** de cada fuente no tenga duplicados.

In [34]:
llave_madre = ["Identificación Cuestionario Individual", "Número de orden en el hogar"]
llave_nino = ["Identificación Cuestionario Individual", "Número de orden en el hogar del niño"]

print("Filas duplicadas exactas en madres:", madres.duplicated().sum())
print("Filas duplicadas exactas en niños:", ninos.duplicated().sum())

print("\nDuplicados por llave propia en madres (se espera 0):", madres.duplicated(subset=llave_madre).sum())
print("Duplicados por llave propia en niños (se espera 0):", ninos.duplicated(subset=llave_nino).sum())

print("\n'Duplicados' esperados en niños por la llave de la MADRE (hogares con más de un niño):")
dup_ninos_madre = ninos[ninos.duplicated(subset=["Identificación Cuestionario Individual","Número de orden de la madre en el hogar"], keep=False)]
print(len(dup_ninos_madre), "filas, correspondientes a",
      dup_ninos_madre[["Identificación Cuestionario Individual","Número de orden de la madre en el hogar"]].drop_duplicates().shape[0],
      "madres distintas con más de un niño en la base — esto NO es un error, es información real.")

Filas duplicadas exactas en madres: 0
Filas duplicadas exactas en niños: 0

Duplicados por llave propia en madres (se espera 0): 0
Duplicados por llave propia en niños (se espera 0): 0

'Duplicados' esperados en niños por la llave de la MADRE (hogares con más de un niño):
3563 filas, correspondientes a 1753 madres distintas con más de un niño en la base — esto NO es un error, es información real.


## 5. Tratamiento de valores inválidos y estandarización

Con los problemas de calidad ya identificados (sección 3), aplicamos el tratamiento:

1. Convertir los códigos sentinela (`9999`, `999`, `9`) a `NaN` real.
2. Escalar a la unidad real ENDES guarda estas variables con un decimal implícito (ej. `586` = `58.6 kg`).
3. Imputar con la **mediana** (más robusta que la media frente a valores extremos de peso/talla).
4. Traducir códigos categóricos a etiquetas legibles, y tratar los blancos como faltante explícito, no como categoría.

In [35]:
# --- Sentinelas -> NaN ---
madres.loc[madres["Peso de la madre"] == 9999, "Peso de la madre"] = np.nan
madres.loc[madres["Talla de la madre"] == 9999, "Talla de la madre"] = np.nan
madres.loc[madres["Nivel de hemoglobina de la madre"] == 999, "Nivel de hemoglobina de la madre"] = np.nan
madres.loc[madres["Nivel de anemia de la madre"] == 9, "Nivel de anemia de la madre"] = np.nan

ninos.loc[ninos["Peso en kilogramos del niño"] == 9999, "Peso en kilogramos del niño"] = np.nan
ninos.loc[ninos["Altura en centímetros del niño"] == 9999, "Altura en centímetros del niño"] = np.nan
ninos.loc[ninos["Nivel de hemoglobina del niño"] == 999, "Nivel de hemoglobina del niño"] = np.nan
ninos.loc[ninos["Nivel de anemia del niño"] == 9, "Nivel de anemia del niño"] = np.nan

# --- Escalar a unidad real ---
madres["Peso de la madre"] /= 10
madres["Talla de la madre"] /= 10
madres["Nivel de hemoglobina de la madre"] /= 10
ninos["Peso en kilogramos del niño"] /= 10
ninos["Altura en centímetros del niño"] /= 10
ninos["Nivel de hemoglobina del niño"] /= 10

# --- Imputar con la mediana ---
for c in ["Peso de la madre","Talla de la madre","Nivel de hemoglobina de la madre"]:
    madres[c] = madres[c].fillna(madres[c].median())
for c in ["Peso en kilogramos del niño","Altura en centímetros del niño","Nivel de hemoglobina del niño"]:
    ninos[c] = ninos[c].fillna(ninos[c].median())

# --- Categorías legibles ---
ninos["El niño se midió acostado o de pie"] = (
    ninos["El niño se midió acostado o de pie"].astype("string").str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .replace({"1": "Acostado", "2": "De pie", "": pd.NA})
)
ninos["Sexo del niño"] = ninos["Sexo del niño"].replace({1: "Hombre", 2: "Mujer"})
madres["Resultado de medición de la madre"] = madres["Resultado de medición de la madre"].replace({
    0: "Medida", 3: "No presente", 4: "Rechazada", 5: "Parcialmente medida", 6: "Otro"
})
madres["Nivel de anemia de la madre"] = madres["Nivel de anemia de la madre"].map({1:"Grave",2:"Moderada",3:"Leve",4:"Sin anemia"})
ninos["Nivel de anemia del niño"] = ninos["Nivel de anemia del niño"].map({1:"Grave",2:"Moderada",3:"Leve",4:"Sin anemia"})
madres["Nivel educativo de la madre"] = madres["Nivel educativo de la madre"].map({0:"Sin educación",1:"Primaria",2:"Secundaria",3:"Superior",8:"No sabe"})

madres_limpio = madres.drop_duplicates(subset=llave_madre, keep="first")
ninos_limpio = ninos.drop_duplicates(subset=llave_nino, keep="first")

print("madres_limpio:", madres_limpio.shape, "| ninos_limpio:", ninos_limpio.shape)
display(madres_limpio.head())
display(ninos_limpio.head())

madres_limpio: (37679, 10) | ninos_limpio: (19298, 11)


,Año,Identificación Cuestionario Individual,Número de orden en el hogar,Edad de la madre en años,Peso de la madre,Talla de la madre,Resultado de medición de la madre,Nivel de hemoglobina de la madre,Nivel de anemia de la madre,Nivel educativo de la madre
0,2025,434400501,2,26,58.6,154.5,Medida,15.1,Sin anemia,Secundaria
1,2025,434401201,1,38,50.4,151.7,Medida,12.8,Leve,Superior
2,2025,434402601,1,44,57.1,148.7,Medida,15.4,Sin anemia,Primaria
3,2025,434402601,2,22,40.8,148.4,Medida,11.8,Moderada,Secundaria
4,2025,434402601,3,13,33.7,143.9,Medida,14.8,Sin anemia,Primaria


,Año,Identificación Cuestionario Individual,Número de orden en el hogar del niño,Edad en meses del niño,Peso en kilogramos del niño,Altura en centímetros del niño,El niño se midió acostado o de pie,Sexo del niño,Nivel de hemoglobina del niño,Nivel de anemia del niño,Número de orden de la madre en el hogar
0,2025,434400501,3,29,13.1,89.4,De pie,Mujer,12.7,Sin anemia,2
1,2025,434401201,2,13,11.4,76.1,Acostado,Hombre,12.3,Sin anemia,1
2,2025,434402601,4,45,13.9,95.9,De pie,Mujer,14.0,Sin anemia,1
3,2025,434403101,3,37,14.9,95.8,De pie,Hombre,12.8,Sin anemia,993
4,2025,434407401,7,13,8.1,70.3,Acostado,Mujer,12.4,Sin anemia,2


## 6. Plan de integración

`concat` sirve para apilar tablas con la misma estructura; `merge` sirve para emparejar tablas con información complementaria usando una clave común. Aquí `madres_limpio` y `ninos_limpio` tienen columnas distintas y describen personas distintas (madre / hijo), por eso el plan de integración usa `merge`, no `concat`.

Usamos `how="outer"` con `indicator=True` para conservar **todos** los registros de ambas fuentes y ver explícitamente cuáles cruzan y cuáles no, en vez de partir directo de un `inner` que ocultaría esa información.

In [36]:
base_integrada = pd.merge(
    madres_limpio,
    ninos_limpio,
    left_on=["Identificación Cuestionario Individual", "Número de orden en el hogar"],
    right_on=["Identificación Cuestionario Individual", "Número de orden de la madre en el hogar"],
    how="outer",
    suffixes=("_madre", "_nino"),
    indicator=True
)

conteo_cruce = base_integrada["_merge"].value_counts().reset_index()
conteo_cruce.columns = ["resultado_cruce", "cantidad"]
print("Filas totales tras la integración auditada:", base_integrada.shape)
display(conteo_cruce)

Filas totales tras la integración auditada: (40432, 21)


,resultado_cruce,cantidad
0,left_only,21134
1,both,18290
2,right_only,1008


**Cómo leer el resultado:**

* **`both`**: mujer y al menos un niño vinculados correctamente (par madre-hijo completo). Es el subconjunto que usamos para el análisis.
* **`left_only`**: mujeres sin ningún niño vinculado; la mayoría de mujeres de 12-49 años en el hogar no tienen hijos menores de 5 años.
* **`right_only`**: niños sin madre identificada (madre no vive en el hogar, no fue entrevistada, o código especial en el registro).

In [37]:
print("Muestra de registros que NO cruzan:")
no_cruzan = base_integrada[base_integrada["_merge"] != "both"]
no_cruzan[["Identificación Cuestionario Individual","Número de orden en el hogar",
           "Número de orden de la madre en el hogar","Edad de la madre en años",
           "Edad en meses del niño","_merge"]].head(10)

Muestra de registros que NO cruzan:


,Identificación Cuestionario Individual,Número de orden en el hogar,Número de orden de la madre en el hogar,Edad de la madre en años,Edad en meses del niño,_merge
3,434402601,2.0,NaN,22.0,NaN,left_only
4,434402601,3.0,NaN,13.0,NaN,left_only
5,434403101,1.0,NaN,48.0,NaN,left_only
6,434403101,NaN,993.0,NaN,37.0,right_only
7,434404001,2.0,NaN,23.0,NaN,left_only
8,434404601,2.0,NaN,31.0,NaN,left_only
9,434404601,3.0,NaN,14.0,NaN,left_only
13,434500401,1.0,NaN,38.0,NaN,left_only
16,434503801,1.0,NaN,43.0,NaN,left_only
18,434505801,1.0,NaN,35.0,NaN,left_only


In [38]:
base_analisis = base_integrada[base_integrada["_merge"] == "both"].drop(columns=["_merge"]).copy()
print("Base integrada para análisis:", base_analisis.shape)
base_analisis.head()

Base integrada para análisis: (18290, 20)


,Año_madre,Identificación Cuestionario Individual,Número de orden en el hogar,Edad de la madre en años,Peso de la madre,Talla de la madre,Resultado de medición de la madre,Nivel de hemoglobina de la madre,Nivel de anemia de la madre,Nivel educativo de la madre,Año_nino,Número de orden en el hogar del niño,Edad en meses del niño,Peso en kilogramos del niño,Altura en centímetros del niño,El niño se midió acostado o de pie,Sexo del niño,Nivel de hemoglobina del niño,Nivel de anemia del niño,Número de orden de la madre en el hogar
0,2025.0,434400501,2.0,26.0,58.6,154.5,Medida,15.1,Sin anemia,Secundaria,2025.0,3.0,29.0,13.1,89.4,De pie,Mujer,12.7,Sin anemia,2.0
1,2025.0,434401201,1.0,38.0,50.4,151.7,Medida,12.8,Leve,Superior,2025.0,2.0,13.0,11.4,76.1,Acostado,Hombre,12.3,Sin anemia,1.0
2,2025.0,434402601,1.0,44.0,57.1,148.7,Medida,15.4,Sin anemia,Primaria,2025.0,4.0,45.0,13.9,95.9,De pie,Mujer,14.0,Sin anemia,1.0
10,2025.0,434407401,2.0,42.0,83.8,150.5,Medida,13.7,Sin anemia,Secundaria,2025.0,7.0,13.0,8.1,70.3,Acostado,Mujer,12.4,Sin anemia,2.0
11,2025.0,434407901,2.0,30.0,68.2,159.3,Medida,13.4,Leve,Secundaria,2025.0,5.0,17.0,11.5,81.1,Acostado,Hombre,11.2,Leve,2.0


## 7. Registro de decisiones de limpieza e integración

En un proyecto real no basta con limpiar y unir: hay que dejar registradas las decisiones y su justificación.

In [39]:
decisiones = pd.DataFrame([
    ["Llave de integración", "No existe un id único de persona; usar solo el ID de hogar generaba un cruce cartesiano mujer x niño",
     "Llave compuesta: Identificación Cuestionario Individual + número de orden de la madre en el hogar",
     "Es la única forma de emparejar cada niño con SU madre real, no con cualquier mujer del hogar."],
    ["Peso / talla / hemoglobina (madre e hijo)", "Código sentinela 9999 (peso/talla) y 999 (hemoglobina) mezclado con las medidas reales",
     "Convertir sentinela a NaN, escalar (valor/10) e imputar con la mediana",
     "9999/999 no son una medida real; la mediana evita distorsionar el análisis con valores extremos."],
    ["Nivel de anemia (madre e hijo)", "Código 9, sin categoría documentada entre Grave/Moderada/Leve/Sin anemia",
     "Tratar como faltante (NaN)",
     "Un código no documentado no puede asumirse como ninguna de las 4 categorías reales."],
    ["El niño se midió acostado o de pie", "Texto en blanco usado como si fuera una tercera categoría",
     "Convertir blanco a faltante explícito (NA)",
     "Un blanco no es una categoría válida; dejarlo así generaría una categoría fantasma en los conteos."],
    ["Duplicados por llave", "Ninguno detectado en esta versión de los datos",
     "Conservar primer registro como salvaguarda (drop_duplicates keep='first')",
     "Buena práctica aunque no haya casos en esta corrida; protege ante nuevas cargas de datos."],
    ["Tipo de unión (madre-niño)", "Se necesita ver tanto los casos que cruzan como los que no, antes de decidir qué usar para el análisis",
     "Integración auditada con how='outer' + indicator=True, y luego filtrar a 'both' para el análisis",
     "Permite documentar cuánta información se pierde (left_only/right_only) en vez de ocultarla con un inner directo."],
], columns=["elemento", "problema", "decisión", "justificación"])

decisiones

,elemento,problema,decisión,justificación
0,Llave de integración,No existe un id único de persona; usar solo el ID de hogar generaba un cruce...,Llave compuesta: Identificación Cuestionario Individual + número de orden de...,"Es la única forma de emparejar cada niño con SU madre real, no con cualquier..."
1,Peso / talla / hemoglobina (madre e hijo),Código sentinela 9999 (peso/talla) y 999 (hemoglobina) mezclado con las medi...,"Convertir sentinela a NaN, escalar (valor/10) e imputar con la mediana",9999/999 no son una medida real; la mediana evita distorsionar el análisis c...
2,Nivel de anemia (madre e hijo),"Código 9, sin categoría documentada entre Grave/Moderada/Leve/Sin anemia",Tratar como faltante (NaN),Un código no documentado no puede asumirse como ninguna de las 4 categorías ...
3,El niño se midió acostado o de pie,Texto en blanco usado como si fuera una tercera categoría,Convertir blanco a faltante explícito (NA),Un blanco no es una categoría válida; dejarlo así generaría una categoría fa...
4,Duplicados por llave,Ninguno detectado en esta versión de los datos,Conservar primer registro como salvaguarda (drop_duplicates keep='first'),Buena práctica aunque no haya casos en esta corrida; protege ante nuevas car...
5,Tipo de unión (madre-niño),"Se necesita ver tanto los casos que cruzan como los que no, antes de decidir...","Integración auditada con how='outer' + indicator=True, y luego filtrar a 'bo...",Permite documentar cuánta información se pierde (left_only/right_only) en ve...


## 8. Visualizaciones iniciales

Se seleccionaron los gráficos según el tipo de cada variable y su relevancia para el problema de segmentación, no se graficó cada variable disponible para evitar diluir la lectura con comparaciones poco informativas (ej. hemoglobina según sexo, o peso vs. talla, que ya están resumidos en los z-scores oficiales del dataset).

**8.1 Variable categórica (resultado de la integración) — gráfico de barras**

In [40]:
etiquetas_cruce = {
    "left_only": "Solo madre (sin niño vinculado)",
    "both": "Madre-niño vinculados",
    "right_only": "Solo niño (sin madre identificada)"
}

conteo_cruce["resultado_cruce_label"] = conteo_cruce["resultado_cruce"].map(etiquetas_cruce)

fig = px.bar(
    conteo_cruce,
    x="resultado_cruce_label",
    y="cantidad",
    text="cantidad",
    title="Resultado de la integración madre-niño (ENDES 2025)",
    labels={"resultado_cruce_label": "Resultado del cruce", "cantidad": "Cantidad de registros"}
)
fig.show()

**Conclusion del resultado de la integración madre-niño:**

El proceso de integración entre RECH5 y RECH6 vinculó correctamente a 18,290 pares madre-niño, equivalentes al 94.8% de los 19,298 niños originales de RECH6, dejando fuera 21,134 mujeres sin niño menor de 5 en el hogar (esperable dada la cobertura poblacional de RECH5, que encuesta a todas las mujeres del hogar tengan o no hijos pequeños) y 1,008 niños sin madre identificada, probablemente por ausencia de la madre en el hogar entrevistado. Esta última cifra constituye la principal limitación de representatividad del análisis, ya que excluye sistemáticamente a niños potencialmente asociados a estructuras familiares distintas (hogares monoparentales, madres ausentes), lo cual debe declararse explícitamente al momento de generalizar los perfiles de riesgo obtenidos hacia el total de la población infantil del país.

**8.2 Variable numérica continua (hemoglobina del niño) — histograma**

Es la variable central del proyecto; antes de cualquier segmentación hay que conocer su distribución y valores extremos.

In [41]:
fig = px.histogram(
    base_analisis,
    x="Nivel de hemoglobina del niño",
    nbins=30,
    marginal="box",
    title="Distribución de la hemoglobina en niños vinculados a su madre"
)
fig.show()

**Conclusión de la distribución de hemoglobina del niño: **

La distribución de hemoglobina del niño muestra un patrón aproximadamente normal centrado en 11.5-12 g/dl, pero con una asimetría marcada hacia valores bajos (4-9 g/dl) que corresponde justamente a los casos de anemia moderada y grave, confirmado por los múltiples outliers inferiores visibles en el boxplot marginal. Esta cola izquierda no es ruido estadístico, sino evidencia visual de que existe un subgrupo separable de niños en riesgo hemoglobínico, lo que valida la elección de un enfoque de segmentación por clustering en lugar de, por ejemplo, un modelo de regresión que asumiría una relación continua y homogénea entre variables.

**8.3 Numérica vs. categórica (hemoglobina del niño según nivel educativo de la madre) — boxplot**

Prueba una hipótesis real de política pública: si el contexto materno se asocia con el estado del niño.

In [42]:
orden_educacion = ["Sin educación","Primaria","Secundaria","Superior","No sabe"]

fig = px.box(
    base_analisis,
    x="Nivel educativo de la madre",
    y="Nivel de hemoglobina del niño",
    category_orders={"Nivel educativo de la madre": orden_educacion},
    points="outliers",
    title="Hemoglobina del niño según nivel educativo de la madre"
)
fig.show()

**Conclusión de hemoglobina según nivel educativo de la madre:**

Al cruzar la hemoglobina del niño con el nivel educativo de la madre, las medianas resultan prácticamente idénticas entre los cuatro grupos (~12 g/dl en todos los casos), con solo diferencias sutiles: el grupo "Sin educación" presenta el rango intercuartílico más amplio y ningún outlier bajo, mientras que "Primaria", "Secundaria" y "Superior" sí muestran varios outliers por debajo de 8 g/dl. Esto indica que el nivel educativo de la madre, tomado de forma aislada, no es un predictor fuerte de la hemoglobina infantil, y refuerza la necesidad de un análisis multivariado que combine varios factores en simultáneo en lugar de conclusiones basadas en cortes univariados simples.

**8.4 Dos numéricas + una categórica (edad vs. hemoglobina, coloreado por anemia) — dispersión**

Anticipa visualmente los futuros grupos de riesgo del clustering.

In [43]:
fig = px.scatter(
    base_analisis,
    x="Edad en meses del niño",
    y="Nivel de hemoglobina del niño",
    color="Nivel de anemia del niño",
    category_orders={"Nivel de anemia del niño": ["Grave","Moderada","Leve","Sin anemia"]},
    color_discrete_map={"Grave":"darkred","Moderada":"orange","Leve":"gold","Sin anemia":"seagreen"},
    hover_data=["Sexo del niño","Nivel educativo de la madre"],
    title="Edad del niño vs. hemoglobina, por nivel de anemia"
)
fig.show()

**Conclusión de la edad del niño vs. hemoglobina, por nivel de anemia:**

Este gráfico contiene el hallazgo más accionable del análisis exploratorio: los casos de anemia moderada (naranja) y grave (rojo) se concentran casi exclusivamente entre los 6 y 24 meses de vida, y prácticamente desaparecen después de los 30-40 meses, mientras que los niños sin anemia (verde) están presentes en todo el rango etario. Esto coincide con el periodo de introducción de alimentación complementaria y mayor demanda de hierro descrito en la literatura de salud pública, y sugiere que cualquier intervención (suplementación de hierro, seguimiento nutricional) debería priorizarse en esa ventana etaria específica en lugar de aplicarse de manera uniforme a todos los menores de 5 años, un insight directamente útil para la priorización de recursos de entidades como MINSA.

In [44]:
variables_numericas = [
    "Edad de la madre en años", "Peso de la madre", "Talla de la madre",
    "Nivel de hemoglobina de la madre", "Edad en meses del niño",
    "Peso en kilogramos del niño", "Altura en centímetros del niño",
    "Nivel de hemoglobina del niño"
]

correlaciones = base_analisis[variables_numericas].corr().round(2)

fig = px.imshow(
    correlaciones,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Correlación entre variables numéricas (madre e hijo)"
)
fig.show()

**Conclusión de la matriz de correlación:**

La matriz de correlación confirma dos cosas relevantes para el diseño del clustering: primero, que edad, peso y talla del niño están altamente correlacionadas entre sí (0.85-0.94), lo cual es esperable biológicamente pero implica que llevarlas las tres sin reducción dimensional inflaría artificialmente su peso en el modelo, justificando aplicar PCA antes de K-means tal como se planteó en los próximos pasos. Segundo, que la hemoglobina materna es la variable de contexto con mayor asociación a la hemoglobina del niño (0.51), muy por encima del resto de correlaciones con esa variable (0.1-0.27), lo que la posiciona como candidata fuerte para incluirse como insumo del clustering y no solo como variable de validación posterior.

In [45]:
distribucion_anemia = base_analisis["Nivel de anemia del niño"].value_counts().reset_index()
distribucion_anemia.columns = ["nivel_anemia", "cantidad"]
print(distribucion_anemia)

fig = px.bar(
    distribucion_anemia,
    x="nivel_anemia",
    y="cantidad",
    text="cantidad",
    category_orders={"nivel_anemia": ["Grave","Moderada","Leve","Sin anemia"]},
    color="nivel_anemia",
    color_discrete_map={"Grave":"darkred","Moderada":"orange","Leve":"gold","Sin anemia":"seagreen"},
    title="Cantidad de niños según nivel de anemia"
)
fig.show()

  nivel_anemia  cantidad
0   Sin anemia     11636
1         Leve      3910
2     Moderada      1134
3        Grave        20


  **Conclusión de la cantidad de niños según nivel de anemia:**

  La distribución de niños por nivel de anemia está fuertemente desbalanceada: 60.3% sin anemia (11,636), 20.2% leve (3,910), 5.9% moderada (1,134) y apenas 0.1% grave (20 casos); además, la suma de estos grupos (16,700) no coincide con los 18,290 pares vinculados, lo que revela alrededor de 1,590 niños sin dato de nivel de anemia registrado.

In [46]:
fig = px.scatter(
    base_analisis,
    x="Nivel de hemoglobina de la madre",
    y="Nivel de hemoglobina del niño",
    color="Nivel de anemia del niño",
    category_orders={"Nivel de anemia del niño": ["Grave","Moderada","Leve","Sin anemia"]},
    color_discrete_map={"Grave":"darkred","Moderada":"orange","Leve":"gold","Sin anemia":"seagreen"},
    trendline="ols",
    title="Hemoglobina de la madre vs. hemoglobina del niño"
)
fig.show()

**Conclusión de Hemoglobina de la madre vs. hemoglobina del niño:**

El scatter entre hemoglobina materna e infantil muestra una tendencia positiva clara dentro de cada nivel de anemia del niño (las cuatro líneas de tendencia suben de izquierda a derecha), pero los grupos aparecen además estratificados verticalmente: los niños con anemia grave y moderada permanecen en las bandas bajas del eje Y casi independientemente del valor de hemoglobina materna, incluso cuando esta es alta (16-20 g/dl). Esto confirma la correlación moderada (0.51) observada en la matriz, pero con un matiz importante: la relación es real y aporta señal, mas no es determinística.

## 9. Hallazgos y conclusiones preliminares


In [48]:
hallazgos = pd.DataFrame({
    "grafico": [
        "Resultado de integración",
        "Distribución de hemoglobina del niño",
        "Hemoglobina según educación de la madre",
        "Edad vs. hemoglobina por nivel de anemia"
    ],
    "observacion": [
        "18,290 pares madre-hijo se vincularon correctamente (both); 21,134 mujeres sin niño vinculado y 1,008 niños sin madre identificada.",
        "La hemoglobina se concentra entre 11 y 13 g/dl, con una cola hacia valores bajos que corresponde a los casos de anemia grave/moderada.",
        "El promedio de hemoglobina del niño es muy similar entre niveles educativos de la madre (~12.0 g/dl en todos los grupos).",
        "Los niños con anemia grave o moderada tienden a ser más pequeños (menor edad promedio) que los niños sin anemia."
    ],
    "hipotesis_preliminar": [
        "El emparejamiento madre-hijo reduce bastante el tamaño de la base útil para el proyecto; hay que declarar esto como limitación de representatividad.",
        "Existe un subgrupo claro de niños con hemoglobina baja, candidato natural para un cluster de 'alto riesgo'.",
        "El nivel educativo de la madre, por sí solo, no parece ser un factor fuerte para explicar la hemoglobina del niño en este corte simple.",
        "La anemia infantil parece concentrarse en los primeros 1-2 años de vida, consistente con la literatura de salud pública."
    ],
    "precaucion": [
        "El análisis final no representa a todas las mujeres/niños de la encuesta, solo al subconjunto que se pudo emparejar.",
        "Un histograma no distingue causas del valor bajo (dieta, altitud, enfermedad); solo describe la distribución.",
        "La ausencia de diferencia visible no prueba que no exista relación; puede estar mediada por otras variables aún no cruzadas.",
        "Relación visual no implica causalidad; la edad también correlaciona con factores no incluidos en este dataset (lactancia, dieta)."
    ]
})

hallazgos

,grafico,observacion,hipotesis_preliminar,precaucion
0,Resultado de integración,"18,290 pares madre-hijo se vincularon correctamente (both); 21,134 mujeres s...",El emparejamiento madre-hijo reduce bastante el tamaño de la base útil para ...,"El análisis final no representa a todas las mujeres/niños de la encuesta, so..."
1,Distribución de hemoglobina del niño,"La hemoglobina se concentra entre 11 y 13 g/dl, con una cola hacia valores b...","Existe un subgrupo claro de niños con hemoglobina baja, candidato natural pa...","Un histograma no distingue causas del valor bajo (dieta, altitud, enfermedad..."
2,Hemoglobina según educación de la madre,El promedio de hemoglobina del niño es muy similar entre niveles educativos ...,"El nivel educativo de la madre, por sí solo, no parece ser un factor fuerte ...",La ausencia de diferencia visible no prueba que no exista relación; puede es...
3,Edad vs. hemoglobina por nivel de anemia,Los niños con anemia grave o moderada tienden a ser más pequeños (menor edad...,"La anemia infantil parece concentrarse en los primeros 1-2 años de vida, con...",Relación visual no implica causalidad; la edad también correlaciona con fact...


## 10. Posibles soluciones y próximos pasos

La salida analítica que persigue este proyecto es una **segmentación** (clustering) de niños según su perfil antropométrico y de anemia, posiblemente precedida de una **reducción dimensional (PCA)** dado que varias variables antropométricas están correlacionadas entre sí.

**Próximos pasos:**
- Estandarizar (media 0, desviación 1) las variables numéricas antes de aplicar K-means o PCA.
- Definir si el nivel de anemia (variable ordinal) se usa como insumo del clustering o se reserva para validar los grupos después.
- Ejecutar K-means con distintos valores de *k*, y evaluar con el método del codo o *silhouette score*.
- Interpretar los clusters resultantes cruzándolos con variables de contexto (educación de la madre, edad del niño).

In [49]:
base_analisis.to_csv("base_integrada_hito1.csv", index=False, encoding="utf-8-sig")
print("Archivo generado: base_integrada_hito1.csv")

Archivo generado: base_integrada_hito1.csv
